# MM Pipeline validation

These checks use fixture records and a fake adapter. They do not call live APIs, Bedrock or S3. Run all cells; a failed assertion stops execution.

In [ ]:
from dataclasses import replace
from datetime import UTC, datetime
from pathlib import Path
from tempfile import TemporaryDirectory
import json
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src' / 'mm_pipeline').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / 'src' / 'mm_pipeline').exists(), 'Run this notebook from mm_pipeline or validation'
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from mm_pipeline.config import PipelineConfig, TopicFilterConfig, load_project_config, resolve_window
from mm_pipeline.models import Article, SourceResult
from mm_pipeline.pipeline import run_pipeline
from mm_pipeline.storage import LocalStorage
from mm_pipeline.transforms import apply_topic_filter, deduplicate_articles, discard_unusable_articles

print('Imports succeeded')

## 1. Validate the supplied project configurations

In [ ]:
ebola_config = load_project_config(PROJECT_ROOT / 'config' / 'projects' / 'ebola.yml')
example_config = load_project_config(PROJECT_ROOT / 'config' / 'projects' / 'example.yml')

assert ebola_config.project.id == 'ebola'
assert any(source.id == 'opoint_news' for source in ebola_config.sources)
assert example_config.project.id == 'respiratory-example'
assert len(ebola_config.enabled_sources) >= 1

print(f'Ebola sources: {len(ebola_config.sources)} total, {len(ebola_config.enabled_sources)} enabled')
print('Configuration checks passed')

## 2. Validate collection-window behaviour

In [ ]:
now = datetime(2026, 10, 27, 12, tzinfo=UTC)
window = resolve_window(ebola_config, lookback_days=7, now=now)

assert window.end == now
assert (window.end - window.start).days == 7
assert window.start.tzinfo is not None and window.end.tzinfo is not None

print(window)
print('Window checks passed')

## 3. Validate filtering, deduplication and empty-text removal

In [ ]:
ebola_article = Article(
    platform='health_source',
    source_id='one',
    published_at=datetime(2026, 10, 25, tzinfo=UTC),
    source_name='Example Health Agency',
    headline='Ebola response expands',
    content='Officials expanded contact tracing in the affected region.',
    summary='',
    url='https://example.test/one',
    language='en',
)
unrelated_health = replace(
    ebola_article, source_id='two', headline='Routine annual report',
    content='No relevant topic is present.', url='https://example.test/two'
)
unrelated_opoint = replace(
    unrelated_health, platform='opoint', source_id='three',
    url='https://example.test/three'
)
empty_article = replace(
    ebola_article, source_id='empty', headline='', content='', summary='',
    url='https://example.test/empty'
)
topic_config = TopicFilterConfig(
    terms=['ebola'], apply_to_platforms=['health_source']
)

filtered = apply_topic_filter(
    [ebola_article, unrelated_health, unrelated_opoint, empty_article],
    topic_config,
)
prepared = discard_unusable_articles(
    deduplicate_articles([*filtered, ebola_article])
)

assert filtered == [ebola_article, unrelated_opoint]
assert prepared == [ebola_article, unrelated_opoint]
assert len({article.article_id for article in prepared}) == len(prepared)

print('Transform checks passed')

## 4. Run the complete pipeline with a fake registered source

In [ ]:
class FakeAdapter:
    def __init__(self, source, logger):
        self.source = source

    def collect(self, collection_window):
        assert collection_window == window
        return SourceResult(self.source.id, [ebola_article])

fake_config = PipelineConfig.model_validate({
    'project': {'id': 'validation-project'},
    'sources': [
        {'id': 'fake', 'adapter': 'fake', 'platform': 'health_source'}
    ],
    'topic_filter': {'enabled': False},
    'output': {'backend': 'local', 'base_uri': 'unused'},
})

with TemporaryDirectory() as temporary_directory:
    result = run_pipeline(
        fake_config,
        window=window,
        adapter_factory=lambda source, logger: FakeAdapter(source, logger),
        storage=LocalStorage(temporary_directory),
    )
    assert result.succeeded
    assert result.articles == [ebola_article]
    master_path = Path(result.output_locations['master_data'])
    manifest_path = Path(result.output_locations['manifest'])
    assert master_path.exists() and manifest_path.exists()
    master_rows = [json.loads(line) for line in master_path.read_text(encoding='utf-8').splitlines()]
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    assert master_rows[0]['article_id'] == ebola_article.article_id
    assert manifest['status'] == 'succeeded'
    assert manifest['records_prepared'] == 1

print('End-to-end fake-source check passed')

## Result

If every cell above completed, the generic configuration and local pipeline path are working. Live adapter and AWS checks should be run separately because they require credentials and network access.